# Sinhala OCR accuracy pilot — Google Colab
Your laptop only needs a browser. This notebook runs **one open-weight OCR model at a time on Google's GPU**, saving results to your Google Drive. It does not train the final model.

**Before opening:** commit/push the code on your current branch. Upload the entire local `reviews/ocr/broad_20261011` folder (including `images`, which Git ignores) to `My Drive/sinhala-corpus/reviews/ocr/broad_20261011`.

Choose **Runtime → Change runtime type → GPU**. For the second-model trial, leave **Surya** selected in the configuration cell. Click the triangle beside each code cell from top to bottom. GPU availability varies; the Surya adapter is not yet GPU validated; earlier Gemma execution and a failed Qwen handwriting trial have supplied Colab evidence. See `docs/COLAB_OCR_GUIDE.md`.

Use only public corpus material. Never upload hidden benchmark questions. Reference files start empty; model outputs remain unreviewed and cannot be treated as accurate training data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Choose OCR backend
Use `gemini_api` for the **best Sinhala OCR quality** (uses the Google Gemini Flash API — free tier: 15 RPM / 1500 pages per day, no GPU needed). Add your API key as `GEMINI_API_KEY` in Colab Secrets.

Other backends: `surya` (local GPU OCR, pins Surya 0.17.1), `gemma` / `qwen` (Vision LLMs, prone to English repetition loops on Sinhala). DeepSeek needs BF16-capable L4/A100 hardware.

In [ ]:
from pathlib import Path
import subprocess, sys, json, os, datetime, shutil
BACKEND = "gemini_api"  # best Sinhala quality; alternatives: surya, gemma, qwen, deepseek
BRANCH = "fix/ocr-fixes"  # change if you later merge the notebook elsewhere
REPOSITORY = "https://github.com/Warusha-Shamalka/sinhala-corpus.git"
ROOT = Path("/content/sinhala-corpus")
DRIVE_ROOT = Path("/content/drive/MyDrive/sinhala-corpus")
PACKET_NAME = "broad_20261011"  # use "pilot_20261011" for the earlier maths-only packet
PACKET = DRIVE_ROOT / "reviews/ocr" / PACKET_NAME
assert (PACKET / "manifest.json").is_file(), "Upload the complete review folder to the Drive path above first"
# GPU check only required for non-API backends
if BACKEND != "gemini_api":
    gpu_tool = shutil.which("nvidia-smi")
    if gpu_tool is None:
        raise RuntimeError(
            "NVIDIA GPU utility is unavailable. In Colab select Runtime > Change runtime type > T4 GPU, "
            "save, reconnect, and rerun cells from the top. GPU backends require an NVIDIA GPU runtime."
        )
    subprocess.run([gpu_tool], check=True)
if ROOT.exists():
    print(f"Cleaning up existing {ROOT} for a fresh clone...")
    shutil.rmtree(ROOT)
print(f"Cloning {REPOSITORY} ({BRANCH})...")
subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPOSITORY,str(ROOT)],check=True)
print("Repository commit:", subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip())

## Install dependencies
For : only lightweight packages are needed (google-generativeai, PyYAML, Pillow). No GPU or heavy ML frameworks required.

For GPU backends (surya/gemma/qwen/deepseek): heavier packages are installed. See previous documentation for details.

If installation fails, stop and share the error. Do not run the whole notebook repeatedly or purchase compute before checking the failure.

In [ ]:
ENV = Path("/content/ocr-env")
if BACKEND == "gemini_api":
    # Clean environment so Google's dependency tree installs fully and is not
    # shadowed by Colab's system "google" namespace packages.
    creation = subprocess.run(
        [sys.executable, "-m", "venv", str(ENV)],
        capture_output=True, text=True,
    )
    if creation.returncode:
        subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
        subprocess.run([sys.executable, "-m", "virtualenv", "--python", sys.executable, str(ENV)], check=True)
elif BACKEND == "surya":
    if ENV.exists():
        raise RuntimeError("Use a fresh runtime for the isolated Surya environment")
    subprocess.run([sys.executable,"-m","pip","install","uv"],check=True)
    subprocess.run([sys.executable,"-m","uv","venv","--python","3.12.12","--seed",str(ENV)],check=True)
else:
    creation = subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(ENV)],
        capture_output=True, text=True,
    )
    if creation.returncode:
        subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
        subprocess.run([sys.executable, "-m", "virtualenv", "--python", sys.executable, "--system-site-packages", str(ENV)], check=True)
PYTHON = str(ENV / "bin/python")
subprocess.run([PYTHON, "-m", "pip", "--version"], check=True)
def pip(*packages):
    subprocess.run([PYTHON,"-m","pip","install",*packages],check=True)
pip("PyYAML==6.0.3","Pillow")
if BACKEND == "gemini_api":
    pip("google-generativeai", "google-api-core", "google-ai-generativelanguage", "google-auth")
    subprocess.run([PYTHON, "-c", "import google.generativeai"], check=True)
elif BACKEND == "surya":
    pip("huggingface_hub")
    pip("torch==2.7.1","torchvision==0.22.1","--index-url","https://download.pytorch.org/whl/cu126")
    pip("surya-ocr==0.17.1","transformers==4.57.6","Pillow==10.4.0")
elif BACKEND in {"gemma", "qwen"}:
    pip("huggingface_hub","--upgrade","transformers","accelerate")
elif BACKEND == "deepseek":
    subprocess.run([PYTHON,"-c","import torch; assert torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported(including_emulation=False)"],check=True)
    pip("huggingface_hub","torch==2.6.0","torchvision==0.21.0","--index-url","https://download.pytorch.org/whl/cu118")
    pip("transformers==4.46.3","tokenizers==0.20.3","einops","easydict","addict","PyMuPDF","img2pdf","numpy","ninja","packaging")
    pip("flash-attn==2.7.3","--no-build-isolation")
else:
    raise ValueError("Unknown backend")
subprocess.run([PYTHON,"-B","-m","pipeline.quality.ocr_benchmark","status",str(PACKET)],cwd=ROOT,check=True)

## API Key / Model setup
For `gemini_api`: Go to [Google AI Studio](https://aistudio.google.com/apikey), create an API key, and add it as `GEMINI_API_KEY` in Colab's **Secrets** panel (key icon). Do not paste the key into a code cell.

For GPU backends: Hugging Face tokens or Surya checkpoints are downloaded as before. See `docs/COLAB_OCR_GUIDE.md`.

In [ ]:
from google.colab import userdata
run_env = os.environ.copy()
RUN = DRIVE_ROOT / "ocr-comparison" / (BACKEND + "_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RUN.mkdir(parents=True, exist_ok=True)
if BACKEND == "gemini_api":
    api_key = None
    for secret_name in ["mmlu", "G_API", "GEMINI_API_KEY", "GOOGLE_API_KEY"]:
        try:
            api_key = userdata.get(secret_name)
            if api_key:
                break
        except Exception:
            pass
    if not api_key:
        raise ValueError("Add your Gemini API key to Colab Secrets with Notebook access turned ON. Key names checked: mmlu, G_API, GEMINI_API_KEY. Get a key at https://aistudio.google.com/apikey")
    run_env["GEMINI_API_KEY"] = api_key
    run_env["G_API"] = api_key
    run_env["mmlu"] = api_key
    MODEL_ID = "gemini-3.8-flash"
    selection = {"model_id": MODEL_ID, "revision": "api", "path": "/content"}
else:
    hf_token = None
    if BACKEND != "surya":
        try:
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    MODEL_ID = {"gemma":"google/gemma-4-E2B-it","deepseek":"deepseek-ai/DeepSeek-OCR-2","qwen":"Qwen/Qwen2.5-VL-3B-Instruct","surya":"datalab/surya-ocr-0.17.1"}[BACKEND]
    if hf_token:
        run_env["HF_TOKEN"] = hf_token
    run_env["HF_HOME"] = "/content/hf-cache"
    selection_code = r"""
import json, sys
from pathlib import Path
model_id, output = sys.argv[1:]
if model_id == "datalab/surya-ocr-0.17.1":
    from pipeline.ocr.surya_backend import prepare_models
    root = Path("/content/surya-models")
    if root.exists():
        from pipeline.ocr.surya_backend import verify_models
        from pipeline.download.downloader import file_hash
        revision = file_hash(root/"selection.json")
        result = {**verify_models(root,revision),"path":str(root),"revision":revision,
                  "revision_kind":"local_components_manifest_sha256"}
    else:
        result = prepare_models(root)
else:
    from huggingface_hub import HfApi, snapshot_download
    revision = HfApi().model_info(model_id).sha
    path = snapshot_download(model_id, revision=revision)
    result = {"model_id":model_id,"revision":revision,"path":path}
Path(output).write_text(json.dumps(result,indent=2)+"\n")
"""
    subprocess.run([PYTHON,"-c",selection_code,MODEL_ID,str(RUN/"model_selection.json")],cwd=ROOT,env=run_env,check=True)
    selection = json.loads((RUN/"model_selection.json").read_text())
(RUN/"requirements.lock.txt").write_text(subprocess.check_output([PYTHON,"-m","pip","freeze"],text=True))
(RUN/"repository_commit.txt").write_text(subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True))
print("Model:",selection.get("model_id", MODEL_ID))
print("Results folder:",RUN)


## Transcribe the broad source images
The runner checks image hashes, never reads reference text, reports each completed page, and saves each prediction immediately to Drive. A new run preserves previous outputs. Start with `MAX_PAGES = 1` for a quick check; after verifying it, set `MAX_PAGES = 14` in a new run folder to transcribe the full broad packet.

For `gemini_api`, the API key is read from Colab Secrets and passed directly to the runner (`--api-key`). For GPU backends (`surya`/`gemma`/`qwen`/`deepseek`), the local model path and pinned revision are passed instead.

Gemma and Qwen have an exact-token repetition-loop guard and a 120-second soft generation deadline; stopped output is preserved as truncated and excluded from accuracy scoring. Surya runs line detection and regional recognition and preserves line/word boxes. DeepSeek upstream `infer` does not use this guard.

In [ ]:
import shutil
from google.colab import userdata

api_key = userdata.get("mmlu") or userdata.get("G_API") or userdata.get("GEMINI_API_KEY")
if BACKEND == "gemini_api":
    assert api_key, "Turn ON Notebook access for secret in Colab Secrets (key icon on left)!"

MAX_PAGES = 1  # set to 1 for quick test, 14 for full broad packet
PREDICTIONS = RUN / "predictions"
if PREDICTIONS.exists():
    shutil.rmtree(PREDICTIONS)

cmd = [
    PYTHON, "-B", "-m", "pipeline.ocr.neural",
    "--packet", str(PACKET),
    "--output", str(PREDICTIONS),
    "--backend", BACKEND,
    "--max-pages", str(MAX_PAGES),
    "--max-new-tokens", "8192",
    "--max-generation-seconds", "120"
]
if BACKEND != "gemini_api":
    cmd.extend(["--model-path", selection["path"], "--revision", selection["revision"]])

subprocess.run(cmd, cwd=ROOT, env={**run_env, "GEMINI_API_KEY": api_key or "", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}, check=True)


## Score only independently verified references
Pending human references produce **pending_references**, not an accuracy estimate. Follow `docs/OCR_ACCURACY_GUIDE.md` to transcribe and independently verify the images before comparing results. The same source document belongs entirely to calibration or holdout.

DeepSeek's upstream API hides generation token counts, so its completion cannot yet be checked automatically. Those outputs remain `completion_unverified` and are excluded from scored results. This is an explicit remaining adapter limitation, not evidence that DeepSeek is inaccurate.

In [ ]:
report = subprocess.check_output([PYTHON,'-B','-m','pipeline.quality.ocr_benchmark',
    'evaluate',str(PACKET),'--predictions',str(PREDICTIONS)],cwd=ROOT,text=True)
(RUN/'accuracy_report.json').write_text(report,encoding='utf-8')
print(report)
print('Saved on Google Drive:',RUN)

## Finish
Inspect the saved page JSON files alongside source images. Do not copy unreviewed output into training datasets. Share the run folder and errors with your team, keeping tokens out of saved outputs. Select **Runtime → Disconnect and delete runtime** when finished; files already saved in Drive remain available.

For a second backend, use a new runtime and change BACKEND. After human references are verified, run the evaluator with two `--predictions` directories to compare identical pages. Do not tune prompts on the holdout pages.